# Reproducible circuit timings

This CPU-sized benchmark measures rotation/nearest-neighbour-CX layers over several circuit widths. Run all cells on your machine after installing qujax and Jupyter. Change CASES and REPEATS to explore larger workloads. The first call includes tracing, compilation and execution; warm calls use an already compiled executable. Every timing waits for device completion.

Memory below is the returned state tensor's payload, not peak process/device memory or compiler workspace. State storage grows exponentially with qubit count. No other simulator is measured, and timings are not a cross-library performance comparison.


In [1]:
import json
import os
import platform
import statistics
from time import perf_counter

import jax
from jax import numpy as jnp
import qujax

metadata = {
    "python": platform.python_version(),
    "system": platform.system(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "qujax": qujax.__version__,
    "jax": jax.__version__,
    "devices": [str(device) for device in jax.devices()],
    "x64_enabled": jax.config.x64_enabled,
    "OMP_NUM_THREADS": os.environ.get("OMP_NUM_THREADS", "unset"),
}
print(json.dumps(metadata, indent=2))


{
  "python": "3.12.13",
  "system": "Windows",
  "machine": "AMD64",
  "processor": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
  "qujax": "1.1.0",
  "jax": "0.6.2",
  "devices": [
    "TFRT_CPU_0"
  ],
  "x64_enabled": false,
  "OMP_NUM_THREADS": "1"
}


In [2]:
CASES = [(n, layers) for n in (4, 6, 8) for layers in (5, 20)]
REPEATS = 15
WARMUPS = 3


def benchmark(n_qubits, layers):
    gates, wires, indices = [], [], []
    for layer in range(layers):
        for qubit in range(n_qubits):
            gates.append("Ry")
            wires.append([qubit])
            indices.append([layer * n_qubits + qubit])
        for qubit in range(n_qubits - 1):
            gates.append("CX")
            wires.append([qubit, qubit + 1])
            indices.append([])
    simulate = jax.jit(
        qujax.get_params_to_statetensor_func(gates, wires, indices, n_qubits=n_qubits)
    )
    params = jax.random.uniform(
        jax.random.PRNGKey(0), (layers * n_qubits,), dtype=jnp.float32
    )
    params.block_until_ready()
    start = perf_counter()
    state = simulate(params)
    state.block_until_ready()
    first_seconds = perf_counter() - start
    for _ in range(WARMUPS):
        simulate(params).block_until_ready()
    durations = []
    for _ in range(REPEATS):
        start = perf_counter()
        state = simulate(params)
        state.block_until_ready()
        durations.append(perf_counter() - start)
    assert bool(jnp.isclose(jnp.sum(jnp.abs(state) ** 2), 1.0, atol=1e-5))
    return {
        "qubits": n_qubits,
        "layers": layers,
        "gates": len(gates),
        "first_call_s": first_seconds,
        "median_warm_ms": 1000 * statistics.median(durations),
        "minimum_warm_ms": 1000 * min(durations),
        "state_bytes": state.nbytes,
        "state_dtype": str(state.dtype),
    }


results = [benchmark(n, layers) for n, layers in CASES]
print(json.dumps(results, indent=2))


[
  {
    "qubits": 4,
    "layers": 5,
    "gates": 35,
    "first_call_s": 1.385067299939692,
    "median_warm_ms": 0.28689997270703316,
    "minimum_warm_ms": 0.1439000479876995,
    "state_bytes": 128,
    "state_dtype": "complex64"
  },
  {
    "qubits": 4,
    "layers": 20,
    "gates": 140,
    "first_call_s": 5.13906789990142,
    "median_warm_ms": 0.37480005994439125,
    "minimum_warm_ms": 0.28220005333423615,
    "state_bytes": 128,
    "state_dtype": "complex64"
  },
  {
    "qubits": 6,
    "layers": 5,
    "gates": 55,
    "first_call_s": 2.635379099752754,
    "median_warm_ms": 0.23639993742108345,
    "minimum_warm_ms": 0.1379000023007393,
    "state_bytes": 512,
    "state_dtype": "complex64"
  },
  {
    "qubits": 6,
    "layers": 20,
    "gates": 220,
    "first_call_s": 5.318668799940497,
    "median_warm_ms": 0.5598999559879303,
    "minimum_warm_ms": 0.43640006333589554,
    "state_bytes": 512,
    "state_dtype": "complex64"
  },
  {
    "qubits": 8,
    "layers":

## Reading the results

Each layer contains one Ry gate per qubit followed by a sequential nearest-neighbour CX chain, so layers are not the transpiled quantum depth. All gate angles are in half turns. Each case uses a fixed random key and float32 parameters; the reported state dtype records actual storage precision.

The first-call number is not a pure compilation measurement. Warm latency includes Python dispatch and device synchronization, which can dominate these small circuits. CPU load, JAX/XLA versions, device and thread configuration affect the result; compare distributions on the same machine rather than treating one table as a guarantee. Raising REPEATS improves the sample size, and larger workloads may require substantially more compilation time and memory.
